# Chapter 3 · Neuroevolution in Rust

**Generate → Evaluate → Select** now works with three representations: flat neural weights, evolving graphs, and compact mutation histories. This original companion was checked against the supplied Chapter 3 PDF on 2026-11-24: Risi, Tang, Ha, and Miikkulainen, *Neuroevolution: Harnessing Creativity in AI Agent Design* (2026), “The Fundamentals of Neuroevolution,” printed pp. 49–71. It uses `nevo` throughout.

Run all cells in order in a fresh Rust/Evcxr kernel, with `notebooks/` as the working directory. The notebook performs real NEAT, classifier, control, and parallel searches. It also reads the larger experiments committed under `book/results/ch03/`. All algorithms, neural computation, physics, and inline plots below are Rust; Python supplies only the Jupyter host.

Read the [chapter](../book/ch03_neuroevolution.md) and [results report](../book/results/ch03/README.md) for the exact variants, costs, sources, and limits. These are teaching implementations, not official exercise solutions or paper-scale reproductions.

| Book section and printed pages | Notebook connection |
| --- | --- |
| §3.1, pp. 49–52 | Flat weights, historical graphs, and seed histories. |
| §3.3.2–3.3.3, pp. 58–62 | NEAT structure, historical crossover, compatibility, and sharing. |
| §3.2, pp. 52–56; §3.3.4, pp. 63–65 | Companion control tasks; the actual BipedalWalkerHardcore and recurrent pole protocols remain unimplemented. |
| §3.4.2–3.4.3, pp. 67–70 | A Deep GA variant and CPU population evaluation; no accelerator or distributed implementation. |

Classification and CartPole also draw on the authors’ separate teaching materials; their recorded experiments are original to this companion.


In [1]:
:dep nevo = { path = "../crates/nevo", features = ["parallel"] }
:dep rayon = "=1.10.0"
:dep plotters = { version = "=0.3.7", default-features = false, features = ["evcxr", "line_series"] }
use nevo::{Algorithm, Evaluated, Evaluator, step};
use nevo::algorithms::*;
use nevo::neat::*;
use nevo::tasks::*;
use nevo::control::*;
use nevo::encoding::{SeedGenome, SeededEncoding};
use nevo::execution::step_parallel;
use nevo::nn::{Activation, Mlp, TrainingExample};
use plotters::prelude::*;


## 1. A graph genome is a reusable value

NEAT labels connections with **historical innovation IDs**. Matching IDs allow crossover to align related genes even when two networks have different structures. These IDs are distinct from evaluation job IDs.

A node-splitting mutation disables one old connection, adds a hidden neuron, and adds connections with weights `1` and `old_weight`. The hidden sigmoid changes the function: this mutation does **not** exactly preserve the old network output. The tracker is shared across a run and deduplicates matching innovations within one generation.


In [2]:
let mut tracker = InnovationTracker::new(2,1)?;
let mut tiny = tracker.initial_genome(vec![0.8,-0.6,0.2])?;
let before = tiny.compile()?.forward(&[0.0,1.0])?[0];
let split_id = tiny.connections()[0].innovation;
tracker.begin_generation();
assert!(split_connection(&mut tiny,&mut tracker,split_id)?);
println!("hidden={}, expressed edges={}",tiny.hidden_node_count(),tiny.enabled_connection_count());
for gene in tiny.connections() {
    println!("innovation {}: {} -> {}, weight={:.3}, enabled={}",
        gene.innovation,gene.from,gene.to,gene.weight,gene.enabled);
}
println!("Same input, before={before:.6}, after={:.6}",tiny.compile()?.forward(&[0.0,1.0])?[0]);


hidden=1, expressed edges=4


innovation 0: 0 -> 3, weight=0.800, enabled=false


innovation 1: 1 -> 3, weight=-0.600, enabled=true


innovation 2: 2 -> 3, weight=0.200, enabled=true


innovation 3: 0 -> 4, weight=1.000, enabled=true


innovation 4: 4 -> 3, weight=0.800, enabled=true


Same input, before=0.123467, after=0.500000


## 2. NEAT uses the same two traits

`generate` produces graph-valued jobs. `NeatXor` compiles each graph and evaluates all four XOR rows. `select` assigns species, shares fitness, updates stagnation and representatives, allocates offspring, and records the best observed genome.

Here is one explicit cycle. Subsequent cycles use `step`, which also checks the executor's nonempty/unique-ID contract. NEAT maximizes nonnegative fitness; for XOR the score is `(4 − sum of absolute prediction errors)²`, with a maximum of 16.


In [3]:
let mut neat_search = Neat::new(NeatConfig::new(2,1,1))?;
let mut neat_task = NeatXor;
let jobs = neat_search.generate()?;
println!("First batch: {} jobs, {} hidden nodes in the first genome",
    jobs.len(),jobs[0].input.hidden_node_count());
let mut evaluated = Vec::with_capacity(jobs.len());
for job in jobs {
    let output = neat_task.evaluate(&job.input,job.seed)?;
    evaluated.push(Evaluated {job,output});
}
neat_search.select(evaluated)?;
println!("Selected batch {}, species={}",neat_search.batches(),neat_search.species().len());


First batch: 150 jobs, 0 hidden nodes in the first genome


Selected batch 1, species=1


Continue until every prediction is within 0.1 of its target, or 300 batches have been selected. Population is 150 and initialization counts toward the budget. Seed 1 is the first successful seed in the saved ten-seed experiment; its success is **not** a claim that every seed succeeds.

This implementation evolves feedforward DAGs only. Hidden/output neurons use `sigmoid(4.9 × weighted_sum)`; the bias input is always 1. The unchanged defaults and all unsuccessful runs are recorded in the results folder.


In [4]:
let xor_targets = [0.0,1.0,1.0,0.0];
let mut neat_history = Vec::<(usize,f64,usize,usize)>::new();
loop {
    let best = neat_search.best().unwrap();
    let predictions = neat_task.predictions(&best.job.input)?;
    neat_history.push((neat_search.evaluations(),best.output,
        best.job.input.hidden_node_count(),neat_search.species().len()));
    let solved = predictions.iter().zip(xor_targets).all(|(a,b)| (a-b).abs() <= 0.1);
    if solved || neat_search.batches() >= 300 { break; }
    step(&mut neat_search,&mut neat_task)?;
}
let xor_genome = neat_search.best().unwrap().job.input.clone();
println!("evaluations={}, hidden={}, enabled connections={}, MSE={:.9}",
    neat_search.evaluations(),xor_genome.hidden_node_count(),
    xor_genome.enabled_connection_count(),neat_task.mse(&xor_genome)?);
for (index,prediction) in neat_task.predictions(&xor_genome)?.iter().enumerate() {
    println!("{} XOR {} -> {:.6}, target {}",index/2,index%2,prediction,xor_targets[index]);
}


evaluations=15900, hidden=4, enabled connections=18, MSE=0.000005889


0 XOR 0 -> 0.001192, target 0


0 XOR 1 -> 0.999843, target 1


1 XOR 0 -> 0.995298, target 1


1 XOR 1 -> 0.000000, target 0


()

In [5]:
let neat_figure = evcxr_figure((1000,400),|root| {
    root.fill(&WHITE)?;
    let panels=root.split_evenly((1,2));
    let end=neat_search.evaluations() as u32;
    let mut fitness=ChartBuilder::on(&panels[0]).caption("NEAT XOR · seed 1",("sans-serif",21))
        .margin(15).x_label_area_size(42).y_label_area_size(48)
        .build_cartesian_2d(0u32..end,0.0f64..16.2f64)?;
    fitness.configure_mesh().x_desc("Fitness evaluations").y_desc("Best observed fitness").draw()?;
    fitness.draw_series(LineSeries::new(neat_history.iter().map(|r|(r.0 as u32,r.1)),&BLUE))?;
    let max_hidden=neat_history.iter().map(|r|r.2).max().unwrap() as u32+1;
    let mut structure=ChartBuilder::on(&panels[1]).caption("Structure of the best observed genome",("sans-serif",20))
        .margin(15).x_label_area_size(42).y_label_area_size(45)
        .build_cartesian_2d(0u32..end,0u32..max_hidden)?;
    structure.configure_mesh().x_desc("Fitness evaluations").y_desc("Hidden node genes").draw()?;
    structure.draw_series(LineSeries::new(neat_history.iter().map(|r|(r.0 as u32,r.2 as u32)),&RGBColor(0,145,120)))?;
    Ok(())
});
neat_figure


NEAT XOR · seed 1
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Best observed fitness
 
 
Fitness evaluations
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
0.0
 
 
 
2.0
 
 
 
4.0
 
 
 
6.0
 
 
 
8.0
 
 
 
10.0
 
 
 
12.0
 
 
 
14.0
 
 
 
16.0
 
 
 
 
0
 
 
 
2000
 
 
 
4000
 
 
 
6000
 
 
 
8000
 
 
 
10000
 
 
 
12000
 
 
 
14000
 
 
<polyline fill="none" opacity="1" stroke="#0000FF" stroke-width="1" points="66,208 70,196 74,191 78,188 82,186 86,182 90,179 94,177 98,177 102,176 106,176 110,176 114,176 118,176 122,176 126,176 130,176 134,176 138,176 142,176 146,176 150,176 154,176 158,176 162,176 166,176 170,176 174,176 178,176 182,176 186,176 190,176 194,176 198,176 202,176 205,176 209,176 213,176 217,176 221,176 225,176 229,176 233,176 237,176 241,176 245,176 249,176 253,176 257,176 261,176 265,176 269,176 273,176 277,176 281,176 285,176 289,176 293,176 297,176 301,176 305,176 309,176 313,176 317,176 321,176 325,176 329,176 333,176 337,176 341,176 344,176 348,176 352,176 356,176 360,176 364,176 368,176 372,176 376,176 380,176 384,176 388,176 392,176 396,176 400,176 404,176 408,176 412,176 416,176 420,176 424,176 428,176 432,176 436,176 440,176 444,176 448,176 452,176 456,176 460,171 464,171 468,171 472,171 476,171 480,167 484,47 "/>
 
Structure of the best observed genome
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Hidden node genes
 
 
Fitness evaluations
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
0
 
 
 
1
 
 
 
2
 
 
 
3
 
 
 
4
 
 
 
5
 
 
 
 
0
 
 
 
2000
 
 
 
4000
 
 
 
6000
 
 
 
8000
 
 
 
10000
 
 
 
12000
 
 
 
14000
 
 
<polyline fill="none" opacity="1" stroke="#009178" stroke-width="1" points="564,342 568,342 572,342 576,342 580,342 584,342 588,342 592,342 596,342 600,342 604,342 608,342 612,342 616,342 620,342 624,342 628,342 632,342 636,342 640,342 644,282 648,342 652,282 656,282 660,282 664,282 668,282 672,282 676,282 680,282 684,282 688,282 692,282 696,282 700,222 704,282 708,222 712,222 716,222 720,222 724,222 728,222 732,282 736,222 740,222 744,222 748,222 752,222 756,282 760,282 764,282 768,282 772,282 776,282 780,282 784,282 788,282 792,282 796,282 800,282 804,282 808,282 812,282 816,282 820,282 824,282 828,282 832,282 836,282 840,282 844,282 848,282 852,282 856,282 860,282 864,282 868,282 872,282 876,282 880,282 884,282 888,282 892,282 896,282 900,282 904,282 908,282 912,282 916,282 920,282 924,282 928,282 932,282 936,282 940,282 944,162 948,162 952,162 956,162 960,162 964,162 968,162 972,162 976,162 980,162 984,102 "/>

In [6]:
let xor_runs=std::fs::read_to_string("../book/results/ch03/neat_xor_runs.csv")?;
let mut xor_solved=0;
for line in xor_runs.lines().skip(1) {
    let c:Vec<_>=line.split(',').collect();
    xor_solved+=usize::from(c[2]=="true");
    println!("seed {:>2}: solved={}, evaluations={}, MSE={:.6}",
        c[0],c[2],c[1],c[4].parse::<f64>()?);
}
println!("Solved {xor_solved}/10; failed runs remain part of the experiment.");


seed  0: solved=false, evaluations=45000, MSE=0.250000


seed  1: solved=true, evaluations=15900, MSE=0.000006


seed  2: solved=true, evaluations=31350, MSE=0.001673


seed  3: solved=false, evaluations=45000, MSE=0.250000


seed  4: solved=true, evaluations=15900, MSE=0.004627


seed  5: solved=true, evaluations=30450, MSE=0.004250


seed  6: solved=false, evaluations=45000, MSE=0.250000


seed  7: solved=true, evaluations=12150, MSE=0.003447


seed  8: solved=true, evaluations=6600, MSE=0.001465


seed  9: solved=true, evaluations=9150, MSE=0.002775


Solved 7/10; failed runs remain part of the experiment.


## 3. Classification: use the correct dataset name

The linked book exercise is labeled “MNIST,” but its starter loads `sklearn.load_digits`: **1,797 images of 8×8 pixels from UCI Optdigits**. This project bundles that original `optdigits.tes` subset with its [CC BY 4.0 attribution and checksum](../data/README.md), and names it accurately.

We divide pixels by 16 and use a seeded stratified split: 60 training images and 30 validation images per class; all remaining images form the test set. That gives **600 / 300 / 897** rows. This teaching split is not the original UCI train/test protocol and is not guaranteed writer-independent.

GA/ES evolve a `64 → 8 tanh → 10 sigmoid` MLP (610 weights/biases). They minimize one-hot MSE. NEAT starts with 650 direct input/bias-to-output links and maximizes `1 − MSE`. Accuracy is a separate argmax metric; the architectures and search settings differ.


In [7]:
let digits_text=std::fs::read_to_string("../data/optdigits.csv")?;
let digits=ClassificationData::from_optdigits_csv(&digits_text)?;
let digits_split=digits.stratified_split(60,30,2026)?;
println!("Samples={}, features={}, classes={}",digits.len(),digits.features(),digits.classes());
println!("Training={}, validation={}, test={}",digits_split.training.len(),digits_split.validation.len(),digits_split.test.len());
let mut digits_task=MlpClassifier::new(digits_split.training.clone(),&[8])?;
println!("Network {:?}: {} parameters",digits_task.widths(),digits_task.dimensions());


Samples=1797, features=64, classes=10


Training=600, validation=300, test=897


Network [64, 8, 10]: 610 parameters


In [8]:
let digit_figure=evcxr_figure((1000,150),|root| {
    root.fill(&WHITE)?;
    for (label,area) in root.split_evenly((1,10)).iter().enumerate() {
        let sample=digits.samples().iter().find(|s|s.label==label).unwrap();
        let mut chart=ChartBuilder::on(area).caption(format!("{label}"),("sans-serif",20))
            .margin(5).build_cartesian_2d(0i32..8i32,0i32..8i32)?;
        chart.draw_series(sample.input.iter().enumerate().map(|(index,value)| {
            let x=(index%8) as i32; let y=7-(index/8) as i32;
            let shade=(255.0*(1.0-value)) as u8;
            Rectangle::new([(x,y),(x+1,y+1)],RGBColor(shade,shade,shade).filled())
        }))?;
    }
    Ok(())
});
digit_figure


0
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
1
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
2
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
3
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
4
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
5
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
6
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
7
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
8
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
9

In [9]:
let mut digits_config=GaConfig::new(digits_task.dimensions(),0);
digits_config.initial_bound=0.5;
digits_config.mutation_sigma=0.2;
let mut digits_search=Ga::new(digits_config)?;
for _ in 0..80 { step(&mut digits_search,&mut digits_task)?; }
let digits_weights=digits_search.best().unwrap().job.input.clone();
for (name,partition) in [("training",&digits_split.training),("validation",&digits_split.validation),("test",&digits_split.test)] {
    let metrics=MlpClassifier::new(partition.clone(),&[8])?.metrics(&digits_weights)?;
    println!("{name:10}: MSE={:.6}, accuracy={:.2}%, samples={}",metrics.mse,100.0*metrics.accuracy,metrics.samples);
}
println!("Training fitness calls={}, samples per call=600",digits_search.evaluations());


training  : MSE=0.081913, accuracy=31.33%, samples=600


validation: MSE=0.081654, accuracy=33.67%, samples=300


test      : MSE=0.081697, accuracy=32.55%, samples=897


Training fitness calls=5120, samples per call=600


The next output reads all three declared seeds for each method. The modest accuracies are the actual results of these small, fixed-budget searches. No holdout score selects a model, a stopping point, or a changed hyperparameter. The validation partition is reserved for later, separately declared development; this run simply reports it after training.


In [10]:
let digits_runs=std::fs::read_to_string("../book/results/ch03/digits_runs.csv")?;
let mut test_accuracy=std::collections::BTreeMap::<String,Vec<f64>>::new();
for line in digits_runs.lines().skip(1) {
    let c:Vec<_>=line.split(',').collect();
    if c[3]=="test" { test_accuracy.entry(c[0].into()).or_default().push(c[6].parse::<f64>()?); }
}
for name in ["ga","es","neat"] {
    let values=&test_accuracy[name];
    let mut ordered=values.clone(); ordered.sort_by(f64::total_cmp);
    println!("{name:4}: test accuracies {:?}%; median {:.2}%",
        values.iter().map(|x|100.0*x).collect::<Vec<_>>(),100.0*ordered[1]);
}


ga  : test accuracies [32.55295429208473, 28.428093645484946, 30.88071348940914]%; median 30.88%


es  : test accuracies [42.80936454849498, 40.802675585284284, 44.25863991081383]%; median 42.81%


neat: test accuracies [43.58974358974359, 59.86622073578596, 48.94091415830547]%; median 48.94%


()

## 4. Control fitness is an aggregate over reset episodes

A controller gets observations and returns actions. The environment advances physics; the evaluator aggregates rewards. `CartPoleTask` resets every candidate to the same three training starts, so chance does not favor one candidate's episodes. The job seed is unused because the trial suite is explicit and fixed.

The book’s §3.2.5 (pp. 55–56) instead trains BipedalWalkerHardcore by averaging **16 random rollouts per candidate**, then evaluates the final agent over **100 random trials**, seeking an average return above 300. Our fixed starts demonstrate aggregation, not that benchmark protocol. The stochastic Gaussian action head mentioned on p. 56 is also outside the current deterministic adapters.

This native CartPole follows the classic equations and discrete ±10 N forces. It receives all four state coordinates and thresholds one tanh output at zero. The native ChaCha resets are not Gym's NumPy random sequence. Reward is +1 per action, including the failure transition; termination and time-limit truncation are separate fields.


In [11]:
let cart_config=CartPoleConfig {max_steps:500,..CartPoleConfig::default()};
let mut cart_env=CartPole::new(cart_config.clone())?;
let cart_starts:Vec<_>=(100..103).map(|seed| {cart_env.reset(seed);*cart_env.state()}).collect();
let mut cart_task=CartPoleTask::new(cart_config.clone(),cart_starts,4)?;
let mut cart_search_config=GaConfig::new(cart_task.dimensions(),0);
cart_search_config.initial_bound=0.5; cart_search_config.mutation_sigma=0.2;
let mut cart_search=Ga::new(cart_search_config)?;
for _ in 0..60 {
    step(&mut cart_search,&mut cart_task)?;
    if -cart_search.best().unwrap().output >= 500.0-1e-9 {break;}
}
let cart_weights=cart_search.best().unwrap().job.input.clone();
let mut cart_returns=Vec::new();
for seed in 200..220 {
    cart_env.reset(seed);
    cart_returns.push(cart_task.rollout(&cart_weights,*cart_env.state())?.total_reward);
}
cart_env.reset(200);
let cart_episode=cart_task.rollout(&cart_weights,*cart_env.state())?;
println!("Fitness calls={}, training mean={:.3}, held-out mean={:.3}",
    cart_search.evaluations(),-cart_search.best().unwrap().output,
    cart_returns.iter().sum::<f64>()/cart_returns.len() as f64);


Fitness calls=512, training mean=500.000, held-out mean=500.000


In [12]:
let cart_figure=evcxr_figure((1000,370),|root| {
    root.fill(&WHITE)?;
    let areas=root.split_evenly((1,2));
    for (panel,title,limit,index) in [(&areas[0],"Cart position",2.4,0usize),(&areas[1],"Pole angle",cart_config.angle_limit,1usize)] {
        let mut chart=ChartBuilder::on(panel).caption(title,("sans-serif",21)).margin(16)
            .x_label_area_size(40).y_label_area_size(55)
            .build_cartesian_2d(0.0f64..10.0f64,-limit*1.1..limit*1.1)?;
        chart.configure_mesh().x_desc("Time (s), held-out start 200")
            .y_desc(if index==0 {"Position (m)"} else {"Angle (rad)"}).draw()?;
        chart.draw_series(LineSeries::new(cart_episode.states.iter().enumerate().map(|(i,s)|
            (i as f64*0.02,if index==0 {s.position}else{s.angle})),&BLUE))?;
        for boundary in [-limit,limit] {
            chart.draw_series(LineSeries::new([(0.0,boundary),(10.0,boundary)],&RED.mix(0.5)))?;
        }
    }
    Ok(())
});
cart_figure


Cart position
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Position (m)
 
 
Time (s), held-out start 200
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-2.5
 
 
 
-2.0
 
 
 
-1.5
 
 
 
-1.0
 
 
 
-0.5
 
 
 
0.0
 
 
 
0.5
 
 
 
1.0
 
 
 
1.5
 
 
 
2.0
 
 
 
2.5
 
 
 
 
0.0
 
 
 
1.0
 
 
 
2.0
 
 
 
3.0
 
 
 
4.0
 
 
 
5.0
 
 
 
6.0
 
 
 
7.0
 
 
 
8.0
 
 
 
9.0
 
 
 
10.0
 
 
<polyline fill="none" opacity="1" stroke="#0000FF" stroke-width="1" points="71,177 71,178 72,178 73,178 74,178 75,179 75,179 76,180 77,180 78,180 79,180 80,181 80,181 81,182 82,182 83,182 84,182 85,182 85,183 86,183 87,183 88,183 89,183 89,183 90,183 91,183 92,183 93,183 94,183 94,183 95,183 96,182 97,182 98,182 99,182 99,182 100,181 101,181 102,180 103,180 103,180 104,179 105,179 106,178 107,178 108,177 108,177 109,176 110,175 111,175 112,174 113,173 113,173 114,172 115,172 116,172 117,171 117,171 118,170 119,170 120,169 121,169 122,169 122,168 123,168 124,168 125,168 126,168 127,168 127,168 128,168 129,168 130,167 131,168 131,168 132,168 133,168 134,168 135,168 136,168 136,169 137,169 138,169 139,170 140,170 141,170 141,171 142,171 143,171 144,172 145,172 145,173 146,173 147,174 148,174 149,175 150,175 150,175 151,176 152,176 153,176 154,177 155,177 155,177 156,178 157,178 158,178 159,178 159,179 160,179 161,179 162,179 163,179 164,179 164,179 165,179 166,179 167,179 168,179 169,178 169,178 170,178 171,178 172,177 173,177 174,177 174,177 175,176 176,176 177,176 178,175 178,175 179,174 180,174 181,173 182,173 183,172 183,172 184,172 185,171 186,171 187,170 188,170 188,169 189,169 190,169 191,168 192,168 192,167 193,167 194,167 195,166 196,166 197,166 197,166 198,166 199,166 200,166 201,165 202,165 202,165 203,165 204,166 205,166 206,166 206,166 207,166 208,166 209,166 210,167 211,167 211,167 212,168 213,168 214,168 215,169 216,169 216,169 217,170 218,170 219,171 220,171 220,172 221,172 222,173 223,173 224,174 225,174 225,174 226,175 227,175 228,176 229,176 230,176 230,176 231,177 232,177 233,177 234,177 234,177 235,178 236,178 237,178 238,178 239,178 239,178 240,178 241,178 242,178 243,177 244,177 244,177 245,177 246,176 247,176 248,175 248,175 249,174 250,174 251,174 252,173 253,173 253,172 254,172 255,171 256,171 257,170 258,170 258,170 259,169 260,169 261,168 262,168 262,167 263,167 264,167 265,166 266,166 267,166 267,165 268,165 269,165 270,165 271,165 272,164 272,164 273,164 274,164 275,165 276,165 277,165 277,165 278,165 279,165 280,165 281,166 281,166 282,166 283,167 284,167 285,167 286,168 286,168 287,168 288,169 289,169 290,170 291,171 291,171 292,172 293,172 294,173 295,173 295,173 296,173 297,174 298,174 299,175 300,175 300,175 301,175 302,176 303,176 304,176 305,176 305,176 306,177 307,177 308,177 309,177 309,177 310,177 311,177 312,177 313,177 314,177 314,176 315,176 316,176 317,176 318,175 319,175 319,175 320,175 321,174 322,174 323,174 323,173 324,173 325,172 326,172 327,171 328,171 328,171 329,170 330,170 331,169 332,169 333,168 333,168 334,167 335,167 336,167 337,167 337,166 338,166 339,166 340,166 341,165 342,165 342,165 343,165 344,165 345,165 346,165 347,166 347,166 348,166 349,166 350,166 351,166 351,166 352,167 353,167 354,167 355,168 356,168 356,168 357,169 358,169 359,169 360,170 361,170 361,170 362,170 363,171 364,171 365,172 365,172 366,173 367,173 368,173 369,173 370,174 370,174 371,175 372,175 373,175 374,175 375,176 375,176 376,177 377,177 378,177 379,177 380,177 380,178 381,178 382,178 383,178 384,178 384,178 385,178 386,178 387,178 388,178 389,177 389,177 390,177 391,177 392,176 393,176 394,176 394,176 395,175 396,175 397,175 398,174 398,174 399,173 400,173 

### Full-state and partial observation are different tasks

`DoublePole` exposes either six coordinates (positions and velocities) or three positions only. A position-only feedforward policy does not gain memory automatically. Caller-owned recurrent state from Chapter 2 can be used in a closure, but recurrent **NEAT** and the historical non-Markov generalization protocol are outside this milestone. The book’s §3.3.4 (pp. 63–65) describes a compact recurrent positions-only solution and a **100,000-step** success criterion. Our **500-step full-state** experiment does not reproduce either.

The saved full-state experiment trains on one start and tests 16 different starts. Both final controllers reach 500 training steps; CMA reaches all 16 held-out horizons, while NEAT reaches 8 and averages 259.375 valid steps. This demonstrates a generalization gap for these particular controllers, not a ranking of the algorithms. Their representations, populations, and stopping costs differ.


In [13]:
let full=DoublePole::new(DoublePoleConfig::default())?;
let partial=DoublePole::new(DoublePoleConfig {observation:PoleObservations::PositionsOnly,..DoublePoleConfig::default()})?;
println!("Full-state observation: {:?}",full.observation());
println!("Position-only observation: {:?}",partial.observation());
let mut uncontrolled=DoublePole::new(DoublePoleConfig::default())?;
let zero_rollout=uncontrolled.rollout(|_|Ok::<_,std::convert::Infallible>(vec![0.0]))?;
println!("Zero-force baseline: {} valid endpoints, {} applied actions, terminated={}",
    zero_rollout.total_reward,zero_rollout.steps,zero_rollout.terminated);
println!("{}",std::fs::read_to_string("../book/results/ch03/doublepole_runs.csv")?);


Full-state observation: [0.0, 0.0, 0.13461538461538464, 0.0, 0.0, 0.0]


Position-only observation: [0.0, 0.13461538461538464, 0.0]


Zero-force baseline: 22 valid endpoints, 23 applied actions, terminated=true


algorithm,seed,batches,fitness_evaluations,training_control_steps,training_valid_steps,training_solved,heldout_mean_valid_steps,heldout_min_valid_steps,heldout_horizon_hits,heldout_trials,hidden_nodes,enabled_connections


cma_es,42,53,1696,117031,5.00000000000000000e2,true,5.00000000000000000e2,5.00000000000000000e2,16,16,4,33


neat,42,9,1350,52831,5.00000000000000000e2,true,2.59375000000000000e2,1.00000000000000000e1,8,16,1,9


## 5. What did the robot actually learn?

`SpringWalker` is an original three-particle, two-actuator spring robot. Actions change leg rest lengths; horizontal movement comes through internal forces and ground contact. Its reward is COM displacement minus positive actuator-work cost and a fall penalty. The book’s §3.2 (pp. 53–54) uses **BipedalWalkerHardcore**: an articulated Box2D robot with **24 observations, four hip/knee actions, and two hidden layers**. That environment and its randomly generated terrain are not implemented here. The spring robot’s return is a different objective and cannot be compared with the book’s 300-point target.

The selected controller was trained for 250 actions on three small torso offsets. Below, we replay the saved **training-selected** weights for 500 actions at neutral friction, then on held-out lower-friction ground. The observed behavior is asymmetric hopping/bounding, not alternating walking. A high reward alone would not tell us that.


In [14]:
let walker_weights:Vec<f64>=std::fs::read_to_string("../book/results/ch03/walker_parameters.txt")?
    .lines().map(str::parse).collect::<Result<_,_>>()?;
let walker_config=SpringWalkerConfig {max_steps:500,..SpringWalkerConfig::default()};
let walker_start=walker_config.initial_state()?;
let walker_task=SpringWalkerTask::new(walker_config.clone(),vec![walker_start],4)?;
let walker_network=walker_task.network(walker_weights)?;
let mut walker_env=SpringWalker::new(walker_config.clone())?;
let walker_episode=walker_env.rollout(|obs|walker_network.forward(obs))?;
let mut slippery_env=SpringWalker::new(SpringWalkerConfig {friction:0.6,..walker_config.clone()})?;
let slippery_episode=slippery_env.rollout(|obs|walker_network.forward(obs))?;
println!("Neutral: steps={}, return={:.5}, distance={:.5} m, work={:.5} J, terminated={}",
    walker_episode.steps,walker_episode.total_reward,walker_env.distance(),walker_env.energy_used(),walker_episode.terminated);
println!("Friction 0.6: steps={}, return={:.5}, distance={:.5} m, work={:.5} J, terminated={}",
    slippery_episode.steps,slippery_episode.total_reward,slippery_env.distance(),slippery_env.energy_used(),slippery_episode.terminated);


Neutral: steps=500, return=35.34788, distance=40.52650 m, work=2589.30675 J, terminated=false


Friction 0.6: steps=222, return=-2.56977, distance=0.53621 m, work=552.99048 J, terminated=true


In [15]:
let motion_figure=evcxr_figure((1000,320),|root| {
    root.fill(&WHITE)?;
    for (area,frame) in root.split_evenly((1,4)).iter().zip([400usize,406,412,418]) {
        let s=walker_episode.states[frame];
        let origin=s.particles[0].position[0];
        let points:Vec<_>=s.particles.iter().map(|p|(p.position[0]-origin,p.position[1])).collect();
        let mut chart=ChartBuilder::on(area).caption(format!("t={:.2} s",frame as f64*0.02),("sans-serif",19))
            .margin(12).x_label_area_size(30).y_label_area_size(35)
            .build_cartesian_2d(-1.4f64..1.5f64,-0.05f64..1.65f64)?;
        chart.configure_mesh().disable_mesh().x_desc("Relative to torso (m)").y_desc("Height (m)").draw()?;
        chart.draw_series(LineSeries::new([(-1.4,0.0),(1.5,0.0)],&BLACK.mix(0.4)))?;
        for (a,b,color) in [(0,1,BLUE),(0,2,RGBColor(225,135,30)),(1,2,RGBColor(130,140,150))] {
            chart.draw_series(LineSeries::new([points[a],points[b]],color.stroke_width(3)))?;
        }
        chart.draw_series(points.iter().enumerate().map(|(i,&p)|Circle::new(p,if i==0 {8}else{5},
            if i==0 {BLACK.filled()} else if i==1 {BLUE.filled()} else {RGBColor(225,135,30).filled()})))?;
    }
    Ok(())
});
motion_figure


t=8.00 s
 
 
Height (m)
 
 
Relative to torso (m)
 
 
 
0.0
 
 
 
0.2
 
 
 
0.4
 
 
 
0.6
 
 
 
0.8
 
 
 
1.0
 
 
 
1.2
 
 
 
1.4
 
 
 
1.6
 
 
 
 
-1.0
 
 
 
-0.5
 
 
 
0.0
 
 
 
0.5
 
 
 
1.0
 
 
 
1.5
 
 
 
 
 
 
 
 
 
 
t=8.12 s
 
 
Height (m)
 
 
Relative to torso (m)
 
 
 
0.0
 
 
 
0.2
 
 
 
0.4
 
 
 
0.6
 
 
 
0.8
 
 
 
1.0
 
 
 
1.2
 
 
 
1.4
 
 
 
1.6
 
 
 
 
-1.0
 
 
 
-0.5
 
 
 
0.0
 
 
 
0.5
 
 
 
1.0
 
 
 
1.5
 
 
 
 
 
 
 
 
 
 
t=8.24 s
 
 
Height (m)
 
 
Relative to torso (m)
 
 
 
0.0
 
 
 
0.2
 
 
 
0.4
 
 
 
0.6
 
 
 
0.8
 
 
 
1.0
 
 
 
1.2
 
 
 
1.4
 
 
 
1.6
 
 
 
 
-1.0
 
 
 
-0.5
 
 
 
0.0
 
 
 
0.5
 
 
 
1.0
 
 
 
1.5
 
 
 
 
 
 
 
 
 
 
t=8.36 s
 
 
Height (m)
 
 
Relative to torso (m)
 
 
 
0.0
 
 
 
0.2
 
 
 
0.4
 
 
 
0.6
 
 
 
0.8
 
 
 
1.0
 
 
 
1.2
 
 
 
1.4
 
 
 
1.6
 
 
 
 
-1.0
 
 
 
-0.5
 
 
 
0.0
 
 
 
0.5
 
 
 
1.0
 
 
 
1.5

## 6. Compact lineage and parallel execution compose with GES

A `SeedGenome` stores one initialization seed plus an ordered list of mutation seeds. The decoder reconstructs a flat parameter vector; the phenotype can be an ordinary `Mlp`. This is reproducible compression by recording how a vector was produced, not a learned developmental encoding like HyperNEAT.

For dimension `d` and `k` mutations, the seed payload is `8 × (k+1)` bytes versus `8 × d` dense bytes. These counts exclude metadata, architecture, decoder settings, and pending-batch copies. Reconstruction costs `O(d × (k+1))`; long histories are not free.


In [16]:
let seed_encoding=SeededEncoding::new(609,0.5,0.1)?;
let lineage=SeedGenome {initial_seed:123,mutation_seeds:vec![456,789]};
let decoded=seed_encoding.decode(&lineage)?;
assert_eq!(decoded.iter().map(|w|w.to_bits()).collect::<Vec<_>>(),
    seed_encoding.decode(&lineage)?.iter().map(|w|w.to_bits()).collect::<Vec<_>>());
println!("Reconstructed {} weights from {} seed payload bytes (dense payload {} bytes)",
    decoded.len(),8*(1+lineage.mutation_seeds.len()),8*decoded.len());


Reconstructed 609 weights from 24 seed payload bytes (dense payload 4872 bytes)


This final experiment evolves a small `2 → 16 → 16 → 16 → 1` network with 609 parameters using the mutation-only Deep GA variant. Selection retains the top 16 of 64 candidates, copies one elite for fresh evaluation, and generates other children by uniform parent sampling plus a dense Gaussian mutation. This follows the mutation/truncation pattern and equation (3.3) in §3.4.2 (p. 68). The noisy Atari protocol there adds **30 episodes for each of the top ten elite contenders**; those extra trials are omitted here. Freshly evaluating our copied elite once is not that contender-selection procedure.

`step_parallel` receives an **evaluator factory**. It creates fresh mutable evaluator state inside each job, collects results in original job order, and calls `select` once. If evaluations fail, it returns the first error in submission order after collecting the jobs and never selects a partial batch. Side effects, thread scheduling, and cross-platform arithmetic are not made deterministic by the runner. This is **CPU task parallelism**; it does not implement the GPU/TPU batching or distributed systems reviewed in §3.4.3 (pp. 69–70).


In [17]:
struct DecodedXor {encoding:SeededEncoding}
impl Evaluator<SeedGenome> for DecodedXor {
    type Output=f64;
    type Error=std::io::Error;
    fn evaluate(&mut self,genes:&SeedGenome,_seed:u64)->Result<f64,Self::Error> {
        let weights=self.encoding.decode(genes).map_err(std::io::Error::other)?;
        let net=Mlp::new(&[2,16,16,16,1],&[Activation::Tanh,Activation::Tanh,Activation::Tanh,Activation::Sigmoid],weights)
            .map_err(std::io::Error::other)?;
        let data=vec![
            TrainingExample {input:vec![0.,0.],target:vec![0.]},
            TrainingExample {input:vec![0.,1.],target:vec![1.]},
            TrainingExample {input:vec![1.,0.],target:vec![1.]},
            TrainingExample {input:vec![1.,1.],target:vec![0.]},
        ];
        net.mse(&data).map_err(std::io::Error::other)
    }
}
fn deep_run(parallel:bool)->Result<(DeepGa,Vec<u64>),Box<dyn std::error::Error>> {
    let mut config=DeepGaConfig::new(609,42); config.initial_bound=0.5;
    let mut search=DeepGa::new(config)?;
    let encoding=*search.encoding();
    let mut evaluator=DecodedXor {encoding};
    let mut scores=Vec::new();
    for _ in 0..100 {
        if parallel {step_parallel(&mut search,||DecodedXor {encoding})?;}
        else {step(&mut search,&mut evaluator)?;}
        let loss=search.best().unwrap().output;
        scores.push(loss.to_bits());
        if loss<=0.01 {break;}
    }
    Ok((search,scores))
}
let (deep_serial,serial_scores)=deep_run(false)?;
let pool=rayon::ThreadPoolBuilder::new().num_threads(4).build()?;
// Build the run inside the pool; return only Send data through its boundary.
let (deep_parallel,parallel_scores)=pool.install(||deep_run(true).map_err(|e|e.to_string()))?;
assert_eq!(serial_scores,parallel_scores);
assert_eq!(deep_serial.best().unwrap(),deep_parallel.best().unwrap());
// Own this result: Evcxr retains top-level bindings between cells.
let deep_best=deep_serial.best().unwrap().clone();
println!("Serial and four-worker runs match: evaluations={}, MSE={:.9}, mutation seeds={}",
    deep_serial.evaluations(),deep_best.output,deep_best.job.input.mutation_seeds.len());
println!("Seed payload={} bytes; dense payload={} bytes",
    8*(1+deep_best.job.input.mutation_seeds.len()),609*8);


Serial and four-worker runs match: evaluations=2304, MSE=0.006776835, mutation seeds=32


Seed payload=264 bytes; dense payload=4872 bytes


## Continue experimenting

- Change one NEAT mutation setting and repeat **all** declared seeds; inspect success rate and topology size.
- Add training starts to the double-pole evaluator, then create a new holdout suite for the next experiment. Reusing already inspected holdouts for development makes them validation data.
- Reward the spring robot for a gait property only if that property matters to your task. Keep displacement, work, contact, and failure diagnostics visible.
- Replace `DecodedXor` with a classifier or control evaluator without changing Deep GA's GES interface.

The CLI experiments reproduce the complete saved records. From the repository root:

```sh
cargo run -p nevo --example ch03_neat --release --locked -- book/results/ch03
cargo run -p nevo --example ch03_classification --release --locked -- book/results/ch03
cargo run -p nevo --example ch03_cartpole --release --locked -- book/results/ch03
cargo run -p nevo --example ch03_doublepole --release --locked -- book/results/ch03
cargo run -p nevo --example ch03_walker --release --locked -- book/results/ch03
cargo run -p nevo --example ch03_parallel --features parallel --release --locked -- book/results/ch03
```

Burn/Bevy adapters, SIMD/tensor batching, GPU execution, distributed orchestration, recurrent NEAT, and historical research-benchmark reproductions remain future work. The scalar algorithms and headless environments provide explicit reference behavior to build on.

### Primary references

Supplied Chapter 3 PDF, printed pp. 49–71 (reviewed 2026-11-24); [book website and separate teaching materials](https://neuroevolutionbook.com/) · [NEAT paper](https://nn.cs.utexas.edu/downloads/papers/stanley.ec02.pdf) · [Deep GA paper](https://arxiv.org/abs/1712.06567) · [UCI Optdigits](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits) · [Gymnasium CartPole](https://gymnasium.farama.org/environments/classic_control/cart_pole/) · [Rayon indexed parallel iteration](https://docs.rs/rayon/1.10.0/rayon/iter/trait.IndexedParallelIterator.html)
